# SODE-Guard — Quickstart

End-to-end walkthrough: synthetic data → train → PGD-40 attack → anti-concentration certificate.
Does **not** require any external dataset download; uses random tensors so the notebook can be executed in CI.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import torch
from src.models.sode_guard import SODEGuard, SODEGuardConfig
from src.attacks.pgd import PGD
from src.certify import model_lipschitz, certify_mean
from src.training.loss import CrossEntropyWithAC
torch.manual_seed(42)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SODEGuard(SODEGuardConfig(num_classes=5, mc_paths_eval=4, certifiable=True)).to(device)
x_train = torch.rand(512, 83, device=device); y_train = torch.randint(0, 5, (512,), device=device)
x_test  = torch.rand( 64, 83, device=device); y_test  = torch.randint(0, 5, ( 64,), device=device)
loss_fn = CrossEntropyWithAC(ac_weight=0.10, n_ac_paths=2)
opt = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
for step in range(20):
    opt.zero_grad(set_to_none=True)
    loss, parts = loss_fn(model, x_train[:64], y_train[:64])
    loss.backward(); opt.step()
print('Final training loss:', parts)

In [ ]:
model.eval()
with torch.no_grad():
    probs = model.forward_mc(x_test); pred = probs.argmax(-1)
print('clean accuracy:', (pred == y_test).float().mean().item())
predict = lambda z: model.forward_mean(z, seeds=range(4))   # the scored 4-path predictor
pgd = PGD(predict, eps=0.03, steps=40)
x_adv = pgd(x_test, y_test)
with torch.no_grad():
    pred_adv = model.forward_mc(x_adv).argmax(-1)
print('PGD-40 ε=0.03 accuracy:', (pred_adv == y_test).float().mean().item())

In [ ]:
# Theorem A: certified l2 radius of the mean predictor (abstains when the margin is too small)
lip = model_lipschitz(model)
cert = certify_mean(model, x_test, L=lip.L, B=20.0, n=1024)
print('L =', round(lip.L, 2), '| certified fraction:', (cert.radius_l2 > 0).float().mean().item(),
      '| median radius among certified:', cert.radius_l2[cert.radius_l2 > 0].median().item() if (cert.radius_l2 > 0).any() else 0.0)